# S1 — CIFAR-100 + Slimmable ResNet-18 + Width

Controlled RQ1 experiment: three anchor-only shared models, twelve independently initialized specialized references, and three representation views. The training horizon is read from a passed S0 artifact; this notebook never defaults to 20 epochs.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
assert torch.cuda.device_count() >= 2, 'Select GPU T4 x2'
for index in range(torch.cuda.device_count()):
    print(f'GPU {index}: {torch.cuda.get_device_name(index)}')

## Secure clone
Create a Kaggle secret named `github_token`.

In [ ]:
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Kaggle secret github_token is missing'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'] if (PROJECT_ROOT / '.git').is_dir() else ['git', 'clone', 'https://github.com/duyh80456-code/new-pruning.git', str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop('GITHUB_TOKEN_RUNTIME', None)
    github_token = None
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1', 'tabulate>=0.9'], check=True)

## Bind the passed S0 decision

Attach the S0 output dataset. Its JSON/YAML decision must contain `s0_pass: true` and a positive integer `selected_horizon`. Set the exact path below if auto-discovery finds more than one candidate.

In [ ]:
from s1_width import read_s0_selection
input_root = Path('/kaggle/input')
names = {'s0_selection.json', 's0_horizon_selection.json', 's0_selection.yaml', 's0_selection.yml'}
candidates = sorted(path for path in input_root.rglob('*') if path.name in names)
assert len(candidates) == 1, f'Expected exactly one S0 selection artifact, found: {candidates}'
S0_SELECTION_PATH = candidates[0]
S0_SELECTION = read_s0_selection(S0_SELECTION_PATH)
print('Locked S0 decision:', S0_SELECTION)

## Preflight and freeze this S1 run

In [ ]:
from datetime import datetime, timezone
import yaml
from torchvision import datasets
SOURCE_CONFIG = PROJECT_ROOT / 'configs' / 'kaggle_s1_width.yaml'
config = yaml.safe_load(SOURCE_CONFIG.read_text())
assert config['training']['epochs'] is None
assert config['experiment']['seeds'] == [0, 1, 2]
assert config['specialization']['widths'] == [0.30, 0.40, 0.60, 0.80]
assert config['dataset']['num_workers'] == 0
datasets.CIFAR100(root=config['dataset']['root'], train=True, download=True)
datasets.CIFAR100(root=config['dataset']['root'], train=False, download=True)
RUN_NAME = datetime.now(timezone.utc).strftime('kaggle-s1-width-%Y%m%d-%H%M%S')
RUN_DIR = Path('/kaggle/working/new-pruning-outputs') / RUN_NAME
config['experiment']['output_dir'] = str(RUN_DIR)
LAUNCH_CONFIG = Path('/kaggle/working/kaggle_s1_width_launch.yaml')
LAUNCH_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('S1 horizon:', S0_SELECTION['selected_horizon'])
print('Run directory:', RUN_DIR)
print('Protocol preflight: OK')

## Execute on two GPUs

Training is resumable at epoch boundaries. The test set stays sealed until all 3 shared and 12 validation-selected specialized checkpoints exist.

In [ ]:
import importlib, s1_width
import scripts.run_s1_width as s1_runner
importlib.reload(s1_width)
s1_runner = importlib.reload(s1_runner)
started = time.perf_counter()
result = s1_runner.run_s1(LAUNCH_CONFIG, S0_SELECTION_PATH, gpu_ids=[0, 1])
print(f'S1 completed in {(time.perf_counter()-started)/3600:.2f} hours')
print(result)

## Inspect the locked outputs

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
display(Markdown((RUN_DIR / 's1_report.md').read_text()))
display(pd.read_csv(RUN_DIR / 'specialization_table.csv'))
display(pd.read_csv(RUN_DIR / 'rq1_representation_correlations.csv'))
display(pd.read_csv(RUN_DIR / 'representation_pattern_robustness.csv'))

In [ ]:
import shutil
archive = Path(shutil.make_archive(str(Path('/kaggle/working') / RUN_NAME), 'zip', root_dir=RUN_DIR))
print('Archive:', archive)
archive